<a href="https://colab.research.google.com/github/Manvit07/health_qa_project/blob/main/dataset_webscraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import requests, re

# The topic file URL changes daily — grab today's link dynamically
listing = requests.get("https://medlineplus.gov/xml.html").text
topics_url = re.search(r'href="(https://medlineplus\.gov/xml/mplus_topics_\d{4}-\d{2}-\d{2}\.xml)"', listing).group(1)
groups_url = re.search(r'href="(https://medlineplus\.gov/xml/mplus_topic_groups_\d{4}-\d{2}-\d{2}\.xml)"', listing).group(1)
print(topics_url)
print(groups_url)

topics_xml = requests.get(topics_url, headers={"User-Agent": "research-prototype/1.0"}).content
groups_xml = requests.get(groups_url, headers={"User-Agent": "research-prototype/1.0"}).content

with open("mplus_topics.xml", "wb") as f: f.write(topics_xml)
with open("mplus_groups.xml", "wb") as f: f.write(groups_xml)

https://medlineplus.gov/xml/mplus_topics_2026-07-09.xml
https://medlineplus.gov/xml/mplus_topic_groups_2026-07-09.xml


In [18]:
import xml.etree.ElementTree as ET

groups_root = ET.parse("mplus_groups.xml").getroot()
for g in groups_root.iter("health-topic-group"):
    print(g.get("id"), "-", g.text)

In [19]:
# Inspect the actual structure before assuming tag names
with open("mplus_groups.xml", "r", encoding="utf-8") as f:
    content = f.read()

print(content[:2000])  # first ~2000 chars, raw

<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE health-topic-groups PUBLIC "-//NLM//DTD health-topic-groups //EN" "https://medlineplus.gov/xml/mplus_groups.dtd">
<health-topic-groups total="88" total-english="44" total-spanish="44" date-generated="07/09/2026 02:30:37">
<group language="English" id="1" url="https://medlineplus.gov/cancers.html">Cancers</group>
<group language="Spanish" id="1" url="https://medlineplus.gov/spanish/cancers.html">Cánceres</group>
<group language="English" id="2" url="https://medlineplus.gov/digestivesystem.html">Digestive System</group>
<group language="Spanish" id="2" url="https://medlineplus.gov/spanish/digestivesystem.html">Sistema digestivo</group>
<group language="English" id="3" url="https://medlineplus.gov/childrenandteenagers.html">Children and Teenagers</group>
<group language="Spanish" id="3" url="https://medlineplus.gov/spanish/childrenandteenagers.html">Niños y adolescentes</group>
<group language="English" id="4" url="https://medlineplus.gov/w

In [20]:
groups_root = ET.parse("mplus_groups.xml").getroot()

english_groups = [
    (g.get("id"), g.text)
    for g in groups_root.iter("group")
    if g.get("language") == "English"
]

for gid, name in sorted(english_groups, key=lambda x: int(x[0])):
    print(gid, "-", name)

1 - Cancers
2 - Digestive System
3 - Children and Teenagers
4 - Women
5 - Mental Health and Behavior
6 - Older Adults
7 - Blood, Heart and Circulation
8 - Substance Use and Disorders
9 - Eyes and Vision
10 - Bones, Joints and Muscles
11 - Kidneys and Urinary System
12 - Infections
14 - Brain and Nerves
15 - Lungs and Breathing
16 - Ear, Nose and Throat
17 - Mouth and Teeth
18 - Skin, Hair and Nails
19 - Injuries and Wounds
20 - Pregnancy and Reproduction
21 - Genetics/Birth Defects
22 - Immune System
23 - Endocrine System
24 - Men
25 - Diagnostic Tests
26 - Food and Nutrition
27 - Wellness and Lifestyle
28 - Poisoning, Toxicology, Environmental Health
29 - Safety Issues
30 - Social/Family Issues
31 - Symptoms
32 - Health System
33 - Population Groups
34 - Sexual Health Issues
35 - Complementary and Alternative Therapies
36 - Drug Therapy
37 - Surgery and Rehabilitation
38 - Transplantation and Donation
39 - Fitness and Exercise
40 - Metabolic Problems
41 - Personal Health Issues
42 - F

In [21]:
with open("mplus_topics.xml", "r", encoding="utf-8") as f:
    topics_content = f.read()

# Find the first complete <health-topic>...</health-topic> block
start = topics_content.find("<health-topic ")
end = topics_content.find("</health-topic>", start) + len("</health-topic>")
print(topics_content[start:end])

<health-topic meta-desc="If you are being tested for Type 2 diabetes, your doctor gives you an A1C test. The test is also used to monitor your A1C levels." title="A1C" url="https://medlineplus.gov/a1c.html" id="6308" language="English" date-created="12/22/2015">
<also-called>Glycohemoglobin</also-called>
<also-called>HbA1C</also-called>
<also-called>Hemoglobin A1C test</also-called>
<full-summary>&lt;p&gt;A1C is a blood test for &lt;a href="https://medlineplus.gov/diabetestype2.html"&gt;type 2 diabetes&lt;/a&gt; and &lt;a href="https://medlineplus.gov/prediabetes.html"&gt;prediabetes&lt;/a&gt;. It measures your average blood glucose, or &lt;a href="https://medlineplus.gov/bloodglucose.html"&gt;blood sugar&lt;/a&gt;, level over the past 3 months. Doctors may use the A1C alone or in combination with other diabetes tests to make a diagnosis. They also use the A1C to see how well you are managing your diabetes. This test is different from the blood sugar checks that people with diabetes do

In [22]:
import xml.etree.ElementTree as ET
import re, html, pandas as pd

TARGET_GROUPS = {19, 44, 31, 41, 30, 27, 26, 39, 36, 29, 28, 5}
GROUP_NAMES = {19:"Injuries and Wounds", 44:"Disasters", 31:"Symptoms",
               41:"Personal Health Issues", 30:"Social/Family Issues",
               27:"Wellness and Lifestyle", 26:"Food and Nutrition",
               39:"Fitness and Exercise", 36:"Drug Therapy",
               29:"Safety Issues", 28:"Poisoning, Toxicology, Environmental Health",
               5:"Mental Health and Behavior"}

def strip_html(raw):
    if not raw:
        return ""
    text = re.sub(r"<[^>]+>", " ", raw)      # drop tags
    text = html.unescape(text)                # decode any remaining entities
    return re.sub(r"\s+", " ", text).strip()

records = []
context = ET.iterparse("mplus_topics.xml", events=("end",))
for event, elem in context:
    if elem.tag == "health-topic":
        if elem.get("language") == "English":
            group_ids = {int(g.get("id")) for g in elem.findall("group")}
            hit = group_ids & TARGET_GROUPS
            if hit:
                summary_elem = elem.find("full-summary")
                summary = strip_html(summary_elem.text if summary_elem is not None else "")
                if len(summary) > 50:  # skip empty/stub topics
                    records.append({
                        "id": elem.get("id"),
                        "title": elem.get("title"),
                        "url": elem.get("url"),
                        "meta_desc": elem.get("meta_desc", ""),
                        "groups": ", ".join(GROUP_NAMES[g] for g in hit),
                        "summary": summary,
                    })
        elem.clear()  # free memory as we go

df = pd.DataFrame(records)
print(f"{len(df)} topics matched")
print(df["groups"].value_counts())
df.to_csv("medlineplus_corpus_raw.csv", index=False)

302 topics matched
groups
Food and Nutrition                                                                50
Injuries and Wounds                                                               41
Mental Health and Behavior                                                        37
Symptoms                                                                          32
Poisoning, Toxicology, Environmental Health                                       19
Drug Therapy                                                                      18
Wellness and Lifestyle                                                            17
Safety Issues                                                                     13
Disasters                                                                         11
Social/Family Issues                                                              10
Wellness and Lifestyle, Fitness and Exercise                                       7
Injuries and Wounds, Symptoms          

In [23]:
for group_name in ["Mental Health and Behavior", "Symptoms", "Injuries and Wounds"]:
    subset = df[df["groups"].str.contains(group_name)]
    print(f"\n--- {group_name} ({len(subset)} topics) ---")
    for t in sorted(subset["title"]):
        print(" ", t)

df["word_count"] = df["summary"].str.split().str.len()
print(df["word_count"].describe())


--- Mental Health and Behavior (43 topics) ---
  Alzheimer's Disease
  Antidepressants
  Anxiety
  Attention Deficit Hyperactivity Disorder
  Autism Spectrum Disorder
  Bereavement
  Bipolar Disorder
  Cancer--Living with Cancer
  Child Behavior Disorders
  Child Mental Health
  Compulsive Gambling
  Coping with Chronic Illness
  Coping with Disasters
  Delirium
  Dementia
  Depression
  Developmental Disabilities
  Dual Diagnosis
  Eating Disorders
  How to Improve Mental Health
  Learning Disabilities
  Memory
  Mental Disorders
  Mental Health
  Mild Cognitive Impairment
  Mood Disorders
  Obsessive-Compulsive Disorder
  Older Adult Mental Health
  Panic Disorder
  Personality Disorders
  Phobias
  Post-Traumatic Stress Disorder
  Postpartum Depression
  Prader-Willi Syndrome
  Psychotic Disorders
  Schizophrenia
  Seasonal Affective Disorder
  Self-Harm
  Stress
  Suicide
  Teen Depression
  Teen Development
  Teen Mental Health

--- Symptoms (38 topics) ---
  Abdominal Pain
  Bad

In [24]:
EXCLUDE_TITLES = {
    # Mental Health and Behavior — specific clinical diagnoses, not companion guidance
    "Alzheimer's Disease", "Antidepressants", "Attention Deficit Hyperactivity Disorder",
    "Autism Spectrum Disorder", "Bipolar Disorder", "Cancer--Living with Cancer",
    "Child Behavior Disorders", "Compulsive Gambling", "Delirium", "Dementia",
    "Developmental Disabilities", "Dual Diagnosis", "Eating Disorders",
    "Learning Disabilities", "Mental Disorders", "Mild Cognitive Impairment",
    "Mood Disorders", "Obsessive-Compulsive Disorder", "Panic Disorder",
    "Personality Disorders", "Phobias", "Post-Traumatic Stress Disorder",
    "Postpartum Depression", "Prader-Willi Syndrome", "Psychotic Disorders",
    "Schizophrenia", "Seasonal Affective Disorder",
    # Symptoms — off-topic stub
    "Rare Diseases",
}

df_curated = df[~df["title"].isin(EXCLUDE_TITLES)].reset_index(drop=True)
print(f"{len(df)} -> {len(df_curated)} topics after exclusions")
df_curated.to_csv("medlineplus_corpus_curated.csv", index=False)



302 -> 274 topics after exclusions


In [25]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

resp = requests.get("https://www.cdc.gov/az/a.html", headers=headers, timeout=15)
print(resp.status_code, len(resp.text))
print(resp.text[:1500])

403 382
<HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;www&#46;cdc&#46;gov&#47;az&#47;a&#46;html" on this server.<P>
Reference&#32;&#35;18&#46;90cd2f17&#46;1783672057&#46;2fa269fa
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;90cd2f17&#46;1783672057&#46;2fa269fa</P>
</BODY>
</HTML>



In [26]:
import requests, time, pandas as pd

API = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "research-prototype/1.0 (academic project; contact: your-email@example.com)"}

def get_category_members(category, limit=500):
    members = []
    params = {
        "action": "query", "list": "categorymembers",
        "cmtitle": f"Category:{category}", "cmlimit": limit,
        "cmnamespace": 0,  # articles only, no subcategories/talk pages
        "format": "json"
    }
    while True:
        r = requests.get(API, params=params, headers=HEADERS).json()
        members += [m["title"] for m in r["query"]["categorymembers"]]
        if "continue" in r:
            params.update(r["continue"])
        else:
            break
    return members

first_aid = get_category_members("First_aid")
self_care = get_category_members("Self-care")
print(f"First aid: {len(first_aid)} articles")
print(f"Self care: {len(self_care)} articles")
print(first_aid[:20])
print(self_care[:20])

First aid: 82 articles
Self care: 36 articles
['First aid', 'ABC (medicine)', 'Act+Fast Anti Choking Trainer', 'Aid station', 'Artificial ventilation', 'Asthma trigger', 'AVPU', 'Band-Aid', 'Bee sting', 'BS 8599', 'Buddy aid', 'Burnol', 'Casualties Union', 'Choking', 'Choking rescue training devices', 'Cold compression therapy', 'Coma scale', 'Compeed', 'Cotton pad', 'E-textiles']
['Self-care', 'Activities of daily living', 'Air displacement plethysmography', 'Audio therapy', 'Autodidacticism', 'Autosuggestion', 'Bed rotting', 'Body composition', 'Caerphilly Heart Disease Study', 'Egonomics', 'Emotional well-being', 'Enema', 'Everyday life', 'First aid', 'Healthy digestion', 'Home pregnancy test', 'Human nutrition', 'Lifestyle medicine', 'List of self-help organizations', 'Locus of control']


In [27]:
print(f"=== First aid ({len(first_aid)}) ===")
for t in first_aid:
    print(" ", t)

print(f"\n=== Self care ({len(self_care)}) ===")
for t in self_care:
    print(" ", t)

=== First aid (82) ===
  First aid
  ABC (medicine)
  Act+Fast Anti Choking Trainer
  Aid station
  Artificial ventilation
  Asthma trigger
  AVPU
  Band-Aid
  Bee sting
  BS 8599
  Buddy aid
  Burnol
  Casualties Union
  Choking
  Choking rescue training devices
  Cold compression therapy
  Coma scale
  Compeed
  Cotton pad
  E-textiles
  Early warning system (medical)
  Elastic bandage
  Elastoplast
  Emergency Bandage
  Emergency bleeding control
  Emergency tourniquet
  Epinephrine autoinjector
  First Aid Council of India
  French Civil Protection
  Gauze sponge
  Golden hour (medicine)
  Good Samaritan law
  Head tilt/Chin lift
  Heimlich maneuver
  Henry Heimlich
  In Case of Emergency
  Inadine
  Insect bites and stings
  Jaw-thrust maneuver
  First aid kit
  Life support
  Lifeguard
  Medical glove
  Medical Tactical Training Program
  MedicAlert
  Mental health first aid
  Moulage
  Mouth-to-mouth resuscitation
  NIBHV
  Nitrous oxide (medication)
  Non-pneumatic anti-shock g

In [28]:
print(f"=== First aid ({len(first_aid)}) ===")
for t in first_aid:
    print(" ", t)

print(f"\n=== Self care ({len(self_care)}) ===")
for t4 in self_care:
    print(" ", t)

=== First aid (82) ===
  First aid
  ABC (medicine)
  Act+Fast Anti Choking Trainer
  Aid station
  Artificial ventilation
  Asthma trigger
  AVPU
  Band-Aid
  Bee sting
  BS 8599
  Buddy aid
  Burnol
  Casualties Union
  Choking
  Choking rescue training devices
  Cold compression therapy
  Coma scale
  Compeed
  Cotton pad
  E-textiles
  Early warning system (medical)
  Elastic bandage
  Elastoplast
  Emergency Bandage
  Emergency bleeding control
  Emergency tourniquet
  Epinephrine autoinjector
  First Aid Council of India
  French Civil Protection
  Gauze sponge
  Golden hour (medicine)
  Good Samaritan law
  Head tilt/Chin lift
  Heimlich maneuver
  Henry Heimlich
  In Case of Emergency
  Inadine
  Insect bites and stings
  Jaw-thrust maneuver
  First aid kit
  Life support
  Lifeguard
  Medical glove
  Medical Tactical Training Program
  MedicAlert
  Mental health first aid
  Moulage
  Mouth-to-mouth resuscitation
  NIBHV
  Nitrous oxide (medication)
  Non-pneumatic anti-shock g

In [29]:
import requests, time, pandas as pd, html

API = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "research-prototype/1.0 (academic project; contact: your-email@example.com)"}

KEEP_TITLES = [
    "First aid", "Asthma trigger", "Bee sting", "Choking", "Cold compression therapy",
    "Emergency bleeding control", "Emergency tourniquet", "Epinephrine autoinjector",
    "Heimlich maneuver", "Insect bites and stings", "First aid kit", "Mental health first aid",
    "Mouth-to-mouth resuscitation", "Nosebleed", "Psychological first aid", "Recovery position",
    "Stop, drop and roll", "Wound",
    "Self-care", "Activities of daily living", "Emotional well-being", "Healthy digestion",
    "Home pregnancy test", "Human nutrition", "Lifestyle medicine", "Self-help",
    "Self-help groups for mental health", "Sleep hygiene", "Support group",
]

HIGH_RISK = {"Emergency tourniquet", "Epinephrine autoinjector", "Mouth-to-mouth resuscitation"}

# Fix 1: follow redirects
def fetch_extracts(titles, delay=0.3):
    records = []
    for title in titles:
        params = {
            "action": "query",
            "prop": "extracts|info",
            "explaintext": 1,
            "exsectionformat": "plain",
            "inprop": "url",
            "redirects": 1,          # <-- follow "Home pregnancy test" -> real target etc.
            "titles": title,
            "format": "json",
        }
        r = requests.get(API, params=params, headers=HEADERS).json()
        pages = r.get("query", {}).get("pages", {})
        for pid, page in pages.items():
            extract = page.get("extract", "")
            url = page.get("fullurl", "")
            real_title = page.get("title", title)
            if not extract or len(extract.split()) < 30:
                print(f"  [warn] still thin/missing: {title} -> {real_title} (pageid={pid})")
                continue
            records.append({
                "source": "Wikipedia", "title": real_title, "url": url,
                "requested_as": title,
                "high_risk": real_title in HIGH_RISK,
                "summary": extract.strip(),
                "attribution": f"Adapted from Wikipedia, \"{real_title}\" (CC BY-SA 4.0), {url}",
            })
        time.sleep(delay)
    return records

wiki_records = fetch_extracts(KEEP_TITLES)
df_wiki = pd.DataFrame(wiki_records)
df_wiki["word_count"] = df_wiki["summary"].str.split().str.len()
print(f"{len(df_wiki)} / {len(KEEP_TITLES)} articles retrieved")

# Check what's driving the long tail
print(df_wiki.sort_values("word_count", ascending=False)[["title", "word_count"]].head(5))

29 / 29 articles retrieved
              title  word_count
23  Human nutrition       16714
3           Choking        8096
18        Self-care        5014
0         First aid        4376
17            Wound        4376


In [30]:
def get_sections(title):
    params = {
        "action": "parse", "page": title, "prop": "sections", "format": "json"
    }
    r = requests.get(API, params=params, headers=HEADERS).json()
    return [s["line"] for s in r.get("parse", {}).get("sections", []) if s["toclevel"] == 1]

for t in ["Human nutrition", "Choking", "Self-care", "First aid", "Wound"]:
    print(f"\n=== {t} ===")
    for s in get_sections(t):
        print(" ", s)


=== Human nutrition ===
  Recommended Dietary Allowances
  Nutrients
  Malnutrition
  Other substances
  Intestinal microbiome
  Global nutrition challenges
  International food insecurity and malnutrition
  Nutrition access disparities
  Nutrition policy
  Advice and guidance
  Nutrition for special populations
  History of human nutrition
  Research of nutrition and nutritional science
  See also
  Further reading
  References
  External links

=== Choking ===
  Signs and symptoms
  Causes
  Diagnosis
  Treatment
  Prevention
  Notable cases
  See also
  References
  External links

=== Self-care ===
  History
  Self-care and illness
  Factors influencing self-care
  Measurement of self-care behaviors
  Middle-range theory of self-care of chronic illness
  Monitoring
  Management
  Self-care for caregivers
  Self-care in philosophy
  Self-Care in the Workplace
  Self-Care and Capitalism
  See also
  References
  External links

=== First aid ===
  Early history and warfare
  Common 

In [31]:
SECTION_POLICY = {
    "Human nutrition": {"keep": ["Recommended Dietary Allowances", "Nutrients", "Malnutrition",
                                  "Other substances", "Advice and guidance", "Nutrition for special populations"]},
    "Choking": {"keep": ["Signs and symptoms", "Causes", "Treatment", "Prevention"]},
    "Self-care": {"keep": ["Self-care and illness", "Factors influencing self-care", "Monitoring",
                            "Management", "Self-care for caregivers"]},
    "First aid": {"keep": ["Common emergences that require first aid in humans", "Aims of first aid",
                            "Setting the priorities", "Key basic skills", "First aid kits", "First aid services"]},
    "Wound": {"keep": ["Classification", "Presentation", "Management"]},
}

def trim_to_sections(full_text, title, all_sections, keep_list):
    # Find byte offsets of every top-level section heading in the plain text
    positions = []
    for sec in all_sections:
        idx = full_text.find(f"\n{sec}\n")
        if idx != -1:
            positions.append((idx, sec))
    positions.sort()
    kept_chunks = []
    for i, (idx, sec) in enumerate(positions):
        end = positions[i+1][0] if i+1 < len(positions) else len(full_text)
        if sec in keep_list:
            kept_chunks.append(full_text[idx:end].strip())
    return "\n\n".join(kept_chunks)

all_sections_cache = {t: get_sections(t) for t in SECTION_POLICY}

for i, row in df_wiki.iterrows():
    if row["title"] in SECTION_POLICY:
        trimmed = trim_to_sections(
            row["summary"], row["title"],
            all_sections_cache[row["title"]],
            SECTION_POLICY[row["title"]]["keep"]
        )
        if len(trimmed.split()) > 30:
            df_wiki.at[i, "summary"] = trimmed
        else:
            print(f"  [warn] trim produced too little text for {row['title']}, check manually")

df_wiki["word_count"] = df_wiki["summary"].str.split().str.len()
print(df_wiki.sort_values("word_count", ascending=False)[["title", "word_count"]].head(10))
df_wiki.to_csv("wikipedia_corpus_curated.csv", index=False)

                                 title  word_count
23                     Human nutrition        8397
3                              Choking        7447
17                               Wound        4025
7             Epinephrine autoinjector        3415
26  Self-help groups for mental health        3179
18                           Self-care        2882
10                       First aid kit        2830
0                            First aid        2822
19          Activities of daily living        2313
27                       Sleep hygiene        2246


In [32]:
# 1. Confirm First aid and Wound are actually different content, not a bug
fa_text = df_wiki.loc[df_wiki["title"] == "First aid", "summary"].values[0]
wd_text = df_wiki.loc[df_wiki["title"] == "Wound", "summary"].values[0]
print("Same content?", fa_text == wd_text)
print("\nFirst aid opening:\n", fa_text[:300])
print("\nWound opening:\n", wd_text[:300])

# 2. Sanity-check Choking's trim — print what sections actually got kept
ch_text = df_wiki.loc[df_wiki["title"] == "Choking", "summary"].values[0]
print("\n\nChoking trimmed text, first 500 chars:\n", ch_text[:500])
print("\nChoking trimmed text, last 500 chars:\n", ch_text[-500:])

Same content? False

First aid opening:
 Common emergences that require first aid in humans
List of common situations that require first aid, and information about them (in alphabetical order):


Bleeding

Bleeding or hemorrhage is the escape of blood from veins or arteries.


Cardiac arrest (total stop of heartbeat)

Cardiac arrest is the

Wound opening:
 Classification
Wounds can be broadly classified as either acute or chronic based on time from initial injury and progression through normal stages of wound healing. Both wound types can further be categorized by cause of injury, wound severity/depth, and sterility of the wound bed. Several classific


Choking trimmed text, first 500 chars:
 Signs and symptoms
Choking victims may present very subtly, especially in the setting of long term foreign body aspiration. Cough is seen in 80% of foreign body aspiration cases, and shortness of breath is seen in 25%. People may be unable to speak, attempt to use hand signals to indicate they are

In [33]:
!pip install -q trafilatura
import trafilatura, requests, pandas as pd, time

BASE = "https://www.who.int/news-room/fact-sheets/detail/"
HEADERS = {"User-Agent": "research-prototype/1.0 (academic project; contact: your-email@example.com)"}

CANDIDATE_SLUGS = [
    "physical-activity", "obesity-and-overweight", "falls", "drowning",
    "road-traffic-injuries", "tobacco", "alcohol", "food-safety",
    "ageing-and-health", "elder-abuse", "ultraviolet-radiation",
    "deafness-and-hearing-loss", "suicide", "immunization-coverage",
    "malnutrition", "breastfeeding", "hand-hygiene", "environmental-noise",
]
# Different URL pattern — verified separately
EXTRA_URLS = {
    "healthy-diet": "https://www.who.int/publications/m/item/healthy-diet-factsheet394",
}

HIGH_RISK = {"suicide", "elder-abuse"}

records = []
for slug in CANDIDATE_SLUGS:
    url = BASE + slug
    resp = requests.get(url, headers=HEADERS, timeout=15)
    status = resp.status_code
    if status != 200:
        print(f"  [FAIL {status}] {slug} -> {url}")
        continue
    text = trafilatura.extract(resp.text, include_tables=False, include_comments=False)
    wc = len(text.split()) if text else 0
    print(f"  [OK {status}, {wc}w] {slug}")
    records.append({"source": "WHO", "title": slug.replace("-", " ").title(), "url": url,
                     "high_risk": slug in HIGH_RISK, "summary": text or "", "word_count": wc})
    time.sleep(0.4)

for title, url in EXTRA_URLS.items():
    resp = requests.get(url, headers=HEADERS, timeout=15)
    text = trafilatura.extract(resp.text, include_tables=False, include_comments=False) if resp.status_code == 200 else None
    if text:
        wc = len(text.split())
        print(f"  [OK {resp.status_code}, {wc}w] {title}")
        records.append({"source": "WHO", "title": title.replace("-", " ").title(), "url": url,
                         "high_risk": False, "summary": text, "word_count": wc})
    else:
        print(f"  [FAIL {resp.status_code}] {title}")

df_who = pd.DataFrame(records)
df_who.to_csv("who_corpus_raw.csv", index=False)
print(f"\n{len(df_who)} / {len(CANDIDATE_SLUGS)+len(EXTRA_URLS)} fact sheets retrieved")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.7 MB/s eta 0:00:00
  [OK 200, 1445w] physical-activity
  [OK 200, 2038w] obesity-and-overweight
  [OK 200, 974w] falls
  [OK 200, 1212w] drowning
  [OK 200, 1443w] road-traffic-injuries
  [OK 200, 1578w] tobacco
  [OK 200, 1577w] alcohol
  [OK 200, 2014w] food-safety
  [OK 200, 1110w] ageing-and-health
  [OK 200, 865w] elder-abuse
  [OK 200, 1022w] ultraviolet-radiation
  [OK 200, 1326w] deafness-and-hearing-loss
  [OK 200, 1025w] suicide
  [OK 200, 1710w] immunization-coverage
  [OK 200, 1237w] malnutrition
  [OK 200, 349w] breastfeeding
  [FAIL 404] hand-hygiene -> https://www.who.int/news-room/fact-sheets/detail/hand-hygiene
  [FAIL 404] environmental-noise -> https://www.who.int/news

In [34]:
falls_text = df_who.loc[df_who["title"] == "Falls", "summary"].values[0]
physact_text = df_who.loc[df_who["title"] == "Physical Activity", "summary"].values[0]

print("=== FALLS (full) ===\n", falls_text)
print("\n\n=== PHYSICAL ACTIVITY (full) ===\n", physact_text)

=== FALLS (full) ===
 The problem
Globally, falls are a major public health problem. An estimated 684 000 fatal falls occur each year, making it the second leading cause of unintentional injury death, after road traffic injuries. Over 80% of fall-related fatalities occur in low- and middle-income countries, with regions of the Western Pacific and South East Asia accounting for 60% of these deaths. In all regions of the world, death rates are highest among adults over the age of 60 years.
Though not fatal, approximately 37.3 million falls severe enough to require medical attention occur each year. Globally, falls are responsible for over 38 million DALYs (disability-adjusted life years) lost each year(2), and result in more years lived with disability than transport injury, drowning, burns and poisoning combined.
While nearly 40% of the total DALYs lost due to falls worldwide occurs in children, this measurement may not accurately reflect the impact of fall-related disabilities for olde

In [35]:
import re

def detect_headers(text):
    """Heuristic: short lines, no ending punctuation, likely section titles."""
    headers = []
    for line in text.split("\n"):
        line = line.strip()
        if 2 <= len(line.split()) <= 8 and not line.endswith((".", ":", ",")) and not line.startswith("-"):
            headers.append(line)
    return headers

for _, row in df_who.iterrows():
    print(f"\n=== {row['title']} ===")
    print(detect_headers(row["summary"]))


=== Physical Activity ===
['Key facts', 'How much physical activity is recommended?', 'Levels of physical inactivity globally', 'WHO response']

=== Obesity And Overweight ===
['Key facts', 'Definition of overweight and obesity', 'Children aged between 5–19 years', 'Children under 5 years of age', 'Prevalence of overweight and obesity', 'Causes of overweight and obesity', 'Common health consequences', 'Facing a double burden of malnutrition', 'Prevention and management', 'WHO response']

=== Falls ===
['The problem', 'Who is at risk?', 'For children and adolescents', 'For workers', 'For older people']

=== Drowning ===
['Key facts', 'Risk factors', 'Poverty and inequality', 'Occupational exposure', 'Climate-related risks', 'Transport on water', 'Migration and seeking refuge', 'WHO response']

=== Road Traffic Injuries ===
['Key facts', 'Who is at risk?', 'Socioeconomic status', 'Risk factors', 'The safe system approach: accommodating human error', 'Non-use of motorcycle helmets, seat-

In [36]:
resp = requests.get("https://www.who.int/news-room/fact-sheets/detail/breastfeeding", headers=HEADERS)
print("Final URL after redirects:", resp.url)
print("Page <title>:", re.search(r"<title>(.*?)</title>", resp.text, re.S).group(1)[:200])

Final URL after redirects: https://www.who.int/news-room/fact-sheets/detail/breastfeeding
Page <title>: 
	Detail

